# Review Corporate Bonds

__Task:__ Programmatically get credit yields currently offered by individual tech firms: Nvidia, SpaceX, and Oracle

__Purpose:__

Focus on these two anomalies to spot trading opportunities or stress points:
* Price Discount from Par (last_price): Bonds are issued at a par value of $100. If a 30-year bond is trading at $82, it indicates massive capital destruction for early buyers. Long-duration bonds drop the deepest during sell-offs because their cash flows are locked far into the future.
* Yield Premium Over Treasuries: If you see a SpaceX 2056 bond yielding significantly more than an Nvidia 2056 bond, you are looking directly at the "risk premium" the market is assigning to SpaceX's capital burn rate.

__Background:__

Because corporate bonds trade in decentralized over-the-counter (OTC) markets rather than on a single centralized stock exchange, finding exact, real-time credit yields requires pulling data from specialized fixed-income platforms, such as FINRA.  

__Manual review using web app:__

FINRA Market Data (TRACE)What it is: The FINRA Fixed Income Data Center is the most authoritative public source for corporate bond prices in the U.S. It tracks the Trade Reporting and Compliance Engine (TRACE), logging every major bond transaction. To manually review data: 

* go to the FINRA Corporate and Agency Bond Search
* type in the company name (e.g., "Nvidia" or "Oracle") 
* the table displays the full list of their outstanding bond maturities, coupon rates, and their Yield to Maturity (YTM) based on the most recent secondary market trades.
* focus on these three indicators to understand the exact yield:
  - CUSIP: Every individual bond tranche has a unique 9-digit alphanumeric identifier (e.g., Nvidia’s 2033 note code)
  - Coupon vs. Yield: The coupon is the interest rate the company promised to pay when it first created the bond (e.g., SpaceX's initial tranche rate). The Yield to Maturity (YTM) is the real yield you get today based on how much the bond's market price has dropped or gained
  - "Spread to Treasury". If a bond has a spread of +140 basis points, it means its exact yield is exactly 1.40% higher than a risk-free government bond of the same length.

`[Bond Identifier: CUSIP] ──► [Coupon Rate: Fixed Payout] ──► [Yield to Maturity (YTM): Actual Return]`


__References:__
* Review the documentation: https://developer.finra.org/docs#query_api-api_basics-datasets
* api wrapper: https://github.com/chencindyj/finra_api_queries

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%cd ..

/workspaces/API-macroecon-wrappy


In [2]:
import sys
sys.path.append('/workspaces/API-macroecon-wrappy')

In [49]:
from macroecon_wrappy import Auth, Metric, Measure

import requests
import json
import pandas as pd
import QuantLib as ql


from pathlib import Path
import pprint

secrets_path = Path('/workspaces/API-macroecon-wrappy/SECRETS.yaml')
cache_path = Path('/workspaces/API-macroecon-wrappy/tests/tmp/')
auth = Auth(secrets_path, cache_path)
auth.load_secrets()

True

### Load data

In [19]:
# FMP searches are case-insensitive partial matches
issuers = ["NVIDIA CORPS", "ORACLE CORP", "SPACE EXPLORATION TECHNOLOGIES"]

# 3. Loop through issuers and fetch corporate bond prices
all_bonds_list = []

print("Fetching corporate bond data from FMP via OpenBB...\n")

Fetching corporate bond data from FMP via OpenBB...



In [56]:
# 1. Authenticate with FINRA
auth_url = 'https://finra.org' # Example token endpoint
auth_payload = {
    'client_id': auth.data['finra_api_key'],
    'client_secret': auth.data['finra_api_secret']
}
response = requests.post(auth_url, data=auth_payload)

In [ ]:
def get_finra_token(key, secret):
    """Generates a temporary access token required for data queries."""
    AUTH_URL = "https://ews.fip.finra.org/fip/rest/ews/oauth2/access_token"
    payload = {"grant_type": "client_credentials"}
    try:
        response = requests.post(AUTH_URL, auth=(key, secret), data=payload)
        response.raise_for_status()
        return response.json().get("access_token")
    except requests.exceptions.HTTPError as err:
        print(f"Authentication Failed: {err}")
        return None

access_token = get_finra_token(auth.data['finra_api_key'], auth.data['finra_api_secret'])

In [94]:
def get_oracle_bond_yields(token):
    """Queries the corporate bond data endpoint filtered for Oracle (ORCL)."""
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json"
    }
    
    # Configure the query filter payload matching FINRA's API specifications
    query_payload = {
        "compareFilters": [
            {
                "fieldName": "issuerReportingSymbol", 
                "operator": "EQUALS", 
                "value": "ORCL"
            }
        ],
        "fields": [
            "bondReportingSymbol", 
            "cusipId", 
            "couponRate", 
            "maturityDate", 
            "yieldToMaturity"
        ],
        "limit": 50
    }
    
    try:
        DATA_URL = "https://finra.org"
        response = requests.post(DATA_URL, json=query_payload, headers=headers)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as err:
        print(f"Data Retrieval Failed: {err}")
        return None
    

Get the most volatile long-duration bonds

In [ ]:
from datetime import datetime


combined_bonds_df = get_oracle_bond_yields(access_token)

# --- PREPARATION STEP ---
# Assuming 'combined_bonds_df' is the DataFrame generated from your OpenBB code.
# Let's ensure the maturity dates are processed as datetime objects.
combined_bonds_df['maturity_date'] = pd.to_datetime(combined_bonds_df['maturity_date'])

# --- STEP 1: CALCULATE YEARS TO MATURITY ---
# Quantify the exact number of years remaining to standardize the filter
current_date = datetime.now()
combined_bonds_df['years_to_maturity'] = (combined_bonds_df['maturity_date'] - current_date).dt.days / 365.25

# --- STEP 2: DEFINE THE DURATION FILTER ---
# In bond markets: Short-term (<5 yrs), Medium-term (5-10 yrs), Long-term (>10 yrs), Ultra-Long (>20 yrs)
MIN_YEARS_THRESHOLD = 10.0 

long_duration_bonds = combined_bonds_df[
    combined_bonds_df['years_to_maturity'] >= MIN_YEARS_THRESHOLD
].copy()

# --- STEP 3: SORT BY VOLATILITY POTENTIAL ---
# Sorting descending by years to maturity puts the most volatile 2056 tranches at the very top
long_duration_bonds = long_duration_bonds.sort_values(by='years_to_maturity', ascending=False)

# --- STEP 4: CLEAN AND VIEW MATRIX ---
# Format numbers for better financial scannability
long_duration_bonds['years_to_maturity'] = long_duration_bonds['years_to_maturity'].round(1)
if 'yield' in long_duration_bonds.columns:
    long_duration_bonds['yield'] = long_duration_bonds['yield'].round(2)

display_cols = ['searched_issuer', 'cusip', 'coupon', 'maturity_date', 'years_to_maturity', 'yield', 'last_price']
existing_display_cols = [col for col in display_cols if col in long_duration_bonds.columns]

print(f"--- VOLATILE TECH BONDS (Maturity >= {MIN_YEARS_THRESHOLD} Years) ---")
print(long_duration_bonds[existing_display_cols].to_string(index=False))